In [1]:
!pip install bertopic
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-c

In [2]:
# import data
import pandas as pd

# load
df = pd.read_csv('/content/result_2.csv')

df.head()

,title,stars,text,bin_pelayanan,bin_fasilitas,predicted_labels_1,labels,prob_layanan,prob_fasilitas
0,RS Brayat Minulya,5,telah dirawat di sini selama beberapa hari di ...,1,1,"['pelayanan', 'fasilitas']",positive,0.996950,0.989263
1,RS Brayat Minulya,5,bersih dan rapi petugas peeawatdokter ditanya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.997084,0.988736
2,RS Brayat Minulya,5,terimakasih atas pelayanan dan perawatan ibu s...,1,0,['pelayanan'],positive,0.997248,0.004785
3,RS Brayat Minulya,5,pelayanan diruang yosef bagus perawat ramah ra...,1,0,['pelayanan'],positive,0.998670,0.012462
4,RS Brayat Minulya,5,tempatnya bagusrapi nyaman sekali ruangan nya ...,1,1,"['pelayanan', 'fasilitas']",positive,0.994812,0.991859


In [3]:
# menghapus stopwords
import re

# daftar
stopwords = ['saya', 'yg', 'dan', 'yang', 'di', 'tidak', 'juga', 'utk', 'itu',
             'untuk', 'juga', 'jg', 'nya', 'tapi', 'tp', 'ga', 'gak', 'ngga', 'nggak', 'sy', 'lagi',
             'lg', 'dengan', 'dgn', 'dg', 'ini', 'kami', 'apa', 'ada', 'ke', 'ya', 'dong', 'sgt', 'bgt',
             'karena', 'krn', 'sebagai', 'sbg', 'dari', 'malah']

# function
def remove_stopwords(text):
    if isinstance(text, str):  # make sure string
        pattern = r'\b(?:' + '|'.join(stopwords) + r')\b'
        return re.sub(pattern, '', text, flags=re.IGNORECASE).strip()
    return text


df['text'] = df['text'].apply(remove_stopwords)

In [4]:
# ubah teks menjadi list
texts = df['text'].astype(str).tolist()

In [5]:
# membangun model
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# gunakan embedding model dari IndoBERT
embedding_model = SentenceTransformer("indobenchmark/indobert-base-p1")

# inisialisasi model
topic_model = BERTopic(embedding_model=embedding_model)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
# fit model
topics, probs = topic_model.fit_transform(texts)

In [7]:
# melihat total topic
len(topic_model.get_topics())

108

In [10]:
# melihat 10 topik teratas
topic_model.get_topic_info().head(6)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3265,-1_dokter_rs_pasien_ramah,"[dokter, rs, pasien, ramah, pelayanan, sangat,...",[minus app bisa reservasi masih bisa lewat...
1,0,310,0_bersih_nyaman_ramah_sangat,"[bersih, nyaman, ramah, sangat, rumah, bagus, ...",[rumah sakit bersih bagus pelayanan baik r...
2,1,207,1_perawatnya_dokternya_ramah_pelayanannya,"[perawatnya, dokternya, ramah, pelayanannya, b...","[pelayanannya bagus dokter perawatnya ramah, ..."
3,2,184,2_theresia_ruang_teresia_diruang,"[theresia, ruang, teresia, diruang, dirawat, k...","[sangat bagus ruang theresia, pelayanan ruang..."
4,3,174,3_terimakasih_semoga_kasih_selalu,"[terimakasih, semoga, kasih, selalu, terima, r...",[terimakasih pelayanan kemoterapi para perawa...
5,4,140,4_nyaman_tempat_bersih_lokasi,"[nyaman, tempat, bersih, lokasi, ok, pelayanan...","[tempat nyaman perawat ramah bersih, tempat b..."


In [9]:
# gabungan dengan labels
df['Topic'] = topics

emotion_topic_dist = pd.crosstab(df['Topic'], df['labels'], normalize='index')
print(emotion_topic_dist)

labels  negative   neutral  positive
Topic                               
-1      0.187749  0.067381  0.744870
 0      0.006452  0.003226  0.990323
 1      0.019324  0.004831  0.975845
 2      0.005435  0.038043  0.956522
 3      0.005747  0.022989  0.971264
...          ...       ...       ...
 102    0.000000  0.000000  1.000000
 103    0.000000  0.000000  1.000000
 104    0.090909  0.181818  0.727273
 105    1.000000  0.000000  0.000000
 106    0.000000  0.000000  1.000000

[108 rows x 3 columns]
